# Latihan SKLearn SVM untuk Klasifikasi
**Nama:** Fauzan Ishaq  
**NIM:** 241011400845

## Tujuan
Menerapkan algoritma **Support Vector Machine (SVM)** menggunakan Scikit-Learn untuk mengklasifikasikan data diabetes berdasarkan dataset **Pima Indians Diabetes**.

Dataset ini memiliki 768 baris data, 8 atribut prediktor, dan 1 kolom target:
- `Outcome = 0`: tidak terindikasi diabetes
- `Outcome = 1`: terindikasi diabetes

> Catatan: tautan Notion yang diberikan tidak dapat dibuka langsung dari sesi ini. Berdasarkan materi yang terindeks dari halaman tersebut, latihan SVM menggunakan dataset Pima Indians Diabetes. Notebook ini menyelesaikan alur klasifikasi SVM secara lengkap dengan dataset tersebut.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print("Library berhasil diimpor.")

## 2. Memuat Dataset

In [ ]:
# Dataset Pima Indians Diabetes dari repositori publik
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

kolom = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(url, header=None, names=kolom)

print("Ukuran dataset:", df.shape)
display(df.head())

## 3. Identifikasi Dataset

In [ ]:
print("Informasi dataset:")
df.info()

print("\\nStatistik deskriptif:")
display(df.describe())

print("\\nJumlah masing-masing kelas:")
display(df["Outcome"].value_counts().rename(index={0: "Tidak diabetes", 1: "Diabetes"}))

print("\\nJumlah data duplikat:", df.duplicated().sum())
print("\\nJumlah nilai kosong per kolom:")
display(df.isnull().sum())

### Penjelasan identifikasi
Dataset terdiri dari 8 fitur numerik dan 1 target klasifikasi. Target `Outcome` memiliki dua kelas sehingga termasuk **klasifikasi biner**. Pemeriksaan awal dilakukan untuk mengetahui ukuran data, tipe data, distribusi kelas, nilai kosong, dan duplikasi.

## 4. Eksplorasi Data

In [ ]:
# Visualisasi jumlah data pada setiap kelas
sns.countplot(data=df, x="Outcome")
plt.title("Distribusi Kelas Outcome")
plt.xlabel("Outcome (0 = Tidak diabetes, 1 = Diabetes)")
plt.ylabel("Jumlah data")
plt.show()

# Korelasi antar fitur
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Korelasi Antar Atribut")
plt.show()

## 5. Preprocessing Data
Pada dataset ini, beberapa kolom seperti Glucose, BloodPressure, SkinThickness, Insulin, dan BMI memiliki nilai 0 yang secara medis umumnya tidak masuk akal sebagai pengukuran. Untuk latihan ini, nilai 0 pada kolom tersebut diperlakukan sebagai nilai yang hilang dan diganti dengan median masing-masing kolom.

In [ ]:
df_clean = df.copy()

kolom_nol = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in kolom_nol:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Jumlah nilai kosong setelah preprocessing:")
display(df_clean.isnull().sum())

print("Contoh data setelah preprocessing:")
display(df_clean.head())

## 6. Memisahkan Fitur dan Target

In [ ]:
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

print("Jumlah fitur:", X.shape[1])
print("Jumlah target:", y.shape[0])

## 7. Membagi Data Training dan Testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", X_train.shape[0])
print("Jumlah data testing :", X_test.shape[0])

## 8. Membuat dan Melatih Model SVM
StandardScaler digunakan agar seluruh fitur berada pada skala yang sebanding. Scaler ditempatkan dalam Pipeline supaya proses standardisasi hanya mempelajari parameter dari data training dan menghindari data leakage.

In [ ]:
# SVM dengan kernel Linear
model_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0))
])

model_linear.fit(X_train, y_train)
pred_linear = model_linear.predict(X_test)

print("Model SVM Linear berhasil dilatih.")

## 9. Evaluasi Model SVM Linear

In [ ]:
akurasi_linear = accuracy_score(y_test, pred_linear)

print(f"Akurasi SVM Linear: {akurasi_linear:.4f} ({akurasi_linear*100:.2f}%)")
print("\\nClassification Report:")
print(classification_report(
    y_test, pred_linear,
    target_names=["Tidak diabetes", "Diabetes"],
    zero_division=0
))

cm = confusion_matrix(y_test, pred_linear)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Tidak diabetes", "Diabetes"]
)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - SVM Linear")
plt.show()

## 10. Mencoba SVM dengan Kernel RBF

In [ ]:
model_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

model_rbf.fit(X_train, y_train)
pred_rbf = model_rbf.predict(X_test)

akurasi_rbf = accuracy_score(y_test, pred_rbf)

print(f"Akurasi SVM RBF: {akurasi_rbf:.4f} ({akurasi_rbf*100:.2f}%)")
print("\\nClassification Report:")
print(classification_report(
    y_test, pred_rbf,
    target_names=["Tidak diabetes", "Diabetes"],
    zero_division=0
))

cm_rbf = confusion_matrix(y_test, pred_rbf)
ConfusionMatrixDisplay(
    confusion_matrix=cm_rbf,
    display_labels=["Tidak diabetes", "Diabetes"]
).plot(cmap="Greens")
plt.title("Confusion Matrix - SVM RBF")
plt.show()

## 11. Perbandingan Kernel

In [ ]:
hasil = pd.DataFrame({
    "Kernel": ["Linear", "RBF"],
    "Akurasi": [akurasi_linear, akurasi_rbf]
})

display(hasil)

plt.figure(figsize=(6, 4))
sns.barplot(data=hasil, x="Kernel", y="Akurasi")
plt.ylim(0, 1)
plt.title("Perbandingan Akurasi SVM")
plt.ylabel("Akurasi")
plt.show()

## 12. Kesimpulan
Pada praktikum ini, algoritma Support Vector Machine berhasil diterapkan untuk mengklasifikasikan dataset Pima Indians Diabetes ke dalam dua kelas, yaitu tidak diabetes dan diabetes. Tahapan yang dilakukan meliputi identifikasi dataset, eksplorasi data, preprocessing, pemisahan data training dan testing, standardisasi fitur, pelatihan model SVM, serta evaluasi menggunakan accuracy, precision, recall, F1-score, dan confusion matrix.

Dua kernel yang dibandingkan adalah Linear dan RBF. Kernel Linear membentuk pemisah linear, sedangkan kernel RBF dapat menangani pola hubungan yang lebih kompleks. Kernel dengan nilai akurasi lebih tinggi pada hasil eksekusi dapat dianggap lebih baik berdasarkan metrik akurasi pada pembagian data ini. Namun, pemilihan model sebaiknya juga mempertimbangkan precision, recall, dan F1-score, terutama karena kesalahan klasifikasi pada data kesehatan perlu diperhatikan.

**Catatan:** hasil klasifikasi ini merupakan latihan machine learning, bukan alat diagnosis medis.